# EduPulse — 피처 엔지니어링 실험

lag feature, 이동 평균, 순환 인코딩 등 피처가 모델 성능에 미치는 영향을 실험합니다.

**실험 항목:**
1. 기본 피처 세트 확인
2. Lag 피처별 상관관계 분석
3. 피처 중요도 (XGBoost feature importance)
4. 피처 조합별 MAPE 비교
5. 새 피처 후보 실험

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from edupulse.preprocessing.merger import build_training_dataset
from edupulse.model.xgboost_model import FEATURE_COLUMNS, TARGET_COLUMN

## 1. 학습 데이터 로딩 및 피처 확인

In [ ]:
import os
data_path = '../edupulse/data/warehouse/training_dataset.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print('학습 데이터 생성 중...')
    df = build_training_dataset()

print(f'데이터: {df.shape}')
print(f'\n사용 피처: {FEATURE_COLUMNS}')
print(f'타겟: {TARGET_COLUMN}')

available = [c for c in FEATURE_COLUMNS if c in df.columns]
missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
print(f'\n가용 피처: {available}')
if missing:
    print(f'누락 피처: {missing}')

## 2. Lag 피처 상관관계 분석

In [ ]:
lag_cols = [c for c in df.columns if c.startswith('lag_') or c.startswith('rolling_')]
if lag_cols:
    corr_with_target = df[lag_cols + [TARGET_COLUMN]].corr()[TARGET_COLUMN].drop(TARGET_COLUMN)
    corr_sorted = corr_with_target.sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['green' if v > 0 else 'red' for v in corr_sorted.values]
    ax.barh(corr_sorted.index, corr_sorted.values, color=colors, alpha=0.7)
    ax.set_xlabel('상관계수')
    ax.set_title(f'Lag/Rolling 피처와 {TARGET_COLUMN} 상관관계')
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()
else:
    print('Lag 피처가 없습니다. 전처리 파이프라인을 먼저 실행하세요.')

## 3. XGBoost 피처 중요도

In [ ]:
from xgboost import XGBRegressor

X = df[available].fillna(0)
y = df[TARGET_COLUMN]

model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42, n_jobs=-1)
model.fit(X, y)

importance = pd.Series(model.feature_importances_, index=available).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost 피처 중요도')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 4. 피처 조합별 MAPE 비교

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def evaluate_features(feature_list, df, n_splits=5):
    """주어진 피처 조합으로 XGBoost MAPE 평가."""
    cols = [c for c in feature_list if c in df.columns]
    X = df[cols].fillna(0).values
    y = df[TARGET_COLUMN].values
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    mapes = []
    for train_idx, val_idx in tscv.split(X):
        model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42, n_jobs=-1)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        nonzero = y[val_idx] != 0
        if nonzero.any():
            mape = np.mean(np.abs((y[val_idx][nonzero] - preds[nonzero]) / y[val_idx][nonzero])) * 100
            mapes.append(mape)
    return np.mean(mapes) if mapes else float('nan')

# 실험할 피처 조합
experiments = {
    '전체 피처': available,
    'Lag만': [c for c in available if 'lag' in c or 'rolling' in c],
    'Lag + 계절': [c for c in available if 'lag' in c or 'rolling' in c or 'month' in c],
    '외부 데이터만': [c for c in available if c in ['search_volume', 'job_count']],
    'Lag 제외': [c for c in available if 'lag' not in c and 'rolling' not in c],
}

results = {}
for name, feats in experiments.items():
    if feats:
        mape = evaluate_features(feats, df)
        results[name] = mape
        print(f'{name:20s}: MAPE {mape:.2f}% (피처 {len(feats)}개)')

fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
values = list(results.values())
colors = ['green' if v == min(values) else 'steelblue' for v in values]
ax.barh(names, values, color=colors, alpha=0.8)
ax.set_xlabel('MAPE (%)')
ax.set_title('피처 조합별 XGBoost MAPE')
plt.tight_layout()
plt.show()

## 5. 새 피처 후보 실험

추가 피처를 만들어보고 성능 변화를 확인합니다.

In [ ]:
# 실험: 검색량 변화율
df_exp = df.copy()

if 'search_volume' in df_exp.columns:
    df_exp['search_volume_diff'] = df_exp.groupby('field')['search_volume'].diff().fillna(0)
    
    new_features = available + ['search_volume_diff']
    mape_new = evaluate_features(new_features, df_exp)
    mape_base = results.get('전체 피처', float('nan'))
    
    print(f'기존 MAPE:          {mape_base:.2f}%')
    print(f'+ search_diff MAPE: {mape_new:.2f}%')
    print(f'변화:               {mape_new - mape_base:+.2f}%')
else:
    print('search_volume 컬럼이 없습니다.')

In [ ]:
# 실험: 채용 공고 이동 평균
if 'job_count' in df_exp.columns:
    df_exp['job_count_rolling_4w'] = df_exp.groupby('field')['job_count'].transform(
        lambda x: x.rolling(4, min_periods=1).mean()
    )
    
    new_features2 = available + ['job_count_rolling_4w']
    mape_new2 = evaluate_features(new_features2, df_exp)
    
    print(f'기존 MAPE:               {mape_base:.2f}%')
    print(f'+ job_rolling_4w MAPE:   {mape_new2:.2f}%')
    print(f'변화:                    {mape_new2 - mape_base:+.2f}%')
else:
    print('job_count 컬럼이 없습니다.')